# D1-13 Optional - Contribution analysis by activity classification

Use the `bw` environment and an existing, licensed `ecoinvent-3.12-cutoff` project
containing the database of the same name. This example groups climate impacts for
**1 kWh of low-voltage electricity in Denmark** using actual ISIC metadata.
Change `classification` to `'CPC'` to group by product classification instead.

In [ ]:
import bw2data as bd
import bw2calc as bc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Select the project, dataset, method, and classification system.

In [ ]:
project = 'ecoinvent-3.12-cutoff'
assert project in bd.projects, 'Import your licensed ecoinvent database first.'
bd.projects.set_current(project)
db = bd.Database('ecoinvent-3.12-cutoff')
matches = [a for a in db if a['name'] == 'market for electricity, low voltage'
           and a.get('reference product') == 'electricity, low voltage'
           and a.get('location') == 'DK']
assert len(matches) == 1
activity = matches[0]
method = ('ecoinvent-3.12', 'EF v3.1', 'climate change', 'global warming potential (GWP100)')
classification = 'ISIC rev.4 ecoinvent'  # Or 'CPC'
print(activity, activity.key)

Sum the characterized inventory's columns to get each activity's own impacts, scaled to
the functional unit. Group by its classification, retaining missing labels and negative
scores. These are not the individual activities' full upstream LCA scores.

In [ ]:
lca = bc.LCA({activity: 1}, method)
lca.lci()
lca.lcia()
scores = np.asarray(lca.characterized_inventory.sum(axis=0)).ravel()

groups = {}
for column in np.flatnonzero(scores):
    node = bd.get_node(id=lca.dicts.activity.reversed[int(column)])
    labels = dict(node.get('classifications', []))
    label = labels.get(classification, 'Unclassified')
    groups[label] = groups.get(label, 0) + scores[column]

grouped = pd.Series(groups).sort_values(key=abs, ascending=False)
np.testing.assert_allclose(grouped.sum(), lca.score)
print('Total:', lca.score, bd.Method(method).metadata['unit'])

Show the ten largest groups by absolute contribution and combine the rest into “Other”.

In [ ]:
plot_scores = grouped.iloc[:10].copy()
if len(grouped) > 10:
    plot_scores.loc['Other classifications'] = grouped.iloc[10:].sum()
ax = plot_scores.iloc[::-1].plot.barh(figsize=(12, 6))
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel(f"Contribution [{bd.Method(method).metadata['unit']}] per kWh")
ax.set_title(f'Danish low-voltage electricity: contributions by {classification}')
plt.tight_layout()
plt.show()